[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/minimal_surfaces/minimal_surfaces.ipynb)

# Minimal Surfaces and Harmonic Conjugates

This example computes minimal surfaces spanning fixed boundary wires, together with their harmonic conjugates and Bonnet's isometric family. A soap film is modelled as a triangulated surface whose area no small interior displacement can reduce, so its mean curvature vanishes across the interior. Turning every small tangent piece of a minimal surface by a quarter turn in its own plane fits the pieces together into a second minimal surface called its harmonic conjugate, and turning them by intermediate angles bends one surface into the other without stretching along Bonnet's family.

The term ***simplex gradient*** here means the plane on a triangle whose pairing rises from zero along a facing side to one at the opposite corner, `(sides | patches.inverse()) / 2`. A quarter turn in the triangle's plane is the commutator `sides.commutator(normals.dual())`, which gives both twice the corner's area gradient and the turned differential that integrates to the harmonic conjugate. Pairing the simplex gradients with half the turned sides, `gradients & turned_sides / 2`, forms the cotangent Laplacian whose vertex pull `laplacian * vertices` sums the area gradients, while the bare facing sides cancel across shared interior edges so the conjugate solves from boundary sums alone.

In the vector notation of discrete differential geometry, as laid out by [Pinkall and Polthier (1993)](https://page.mi.fu-berlin.de/polthier/articles/diri/diri_jem.pdf), the area gradient at a triangle's corner is $\nabla_{p_i} A_T = \frac{1}{2} N_T \times e_i$, the cotangent Laplacian is $(L X)_i = \frac{1}{2}\sum_{j} (\cot \alpha_{ij} + \cot \beta_{ij})(X_i - X_j) = -2 H_i A_i N_i$, with entries $L_{ij} = \sum_T A_T\,\nabla\phi_i\cdot\nabla\phi_j$ over the piecewise-linear basis functions $\phi_i$, the conjugate integrates $dX^* = N \times dX$, and Bonnet's family is $X_\theta = \cos\theta\, X + \sin\theta\, X^*$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from numga.sparse import spdiag
from numga.topology import Topology
from examples.surfaces.minimal_surfaces import render, scenarios
from examples.util.animation import save_figure, show_animation

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Point = ga.gatype.antivector()                                               # Antivector-
Patch = ga.gatype.antibivector()                                             # Antibivector-
# Mean curvature, measured against the triangles' own winding.
Curvature = ga.gatype.antipseudoscalar()                                     # Antipseudoscalar-
frame_duration_ms = 40

np.set_printoptions(precision=4, suppress=True)

## Area gradient and soap-film relaxation

Across a soap film, the area gradient vanishes at every interior vertex inside the held boundary wire, here a circle raised into four arches and initially spanned by a dome. Each damped step solves for the harmonic surface of the current triangles, where their cotangent-Laplacian pull vanishes on the interior, and steps partway towards it so total area decreases monotonically.

In [ ]:
disk_radius = 1.0
n_disk_divisions = 10
arch_height = 0.35
dome_height = 0.55
relaxation_step_size = 0.14
n_relaxation_steps = 40

In [ ]:
plateau_topology, flat_vertices, initial_vertices = scenarios.four_arch_dome(
    disk_radius, n_disk_divisions, arch_height, dome_height,
)                                                                            # Topology, Point-[V], Point-[V]
corners = plateau_topology.incidence[2, 0]                                   # [3] Scalar+[F <- V]
# Each facing side runs in the direction of its triangle's winding.
sides = plateau_topology.boundary[2, 1] * (plateau_topology.d[0] * initial_vertices)  # [3] Point-[F]
patches = (sides[0] & sides[1]) / 2                                          # Patch-[F]
areas = patches.norm()                                                       # Scalar+[F]
normals = patches / areas                                                    # Patch-[F]
# Each corner's simplex gradient vanishes along the facing side and rises to one at the corner.
gradients = (sides | patches.inverse()) / 2                                  # [3] Plane+[F]
# Turning each facing side by a quarter turn in the triangle's plane gives twice that corner's area gradient.
turned_sides = sides.commutator(normals.dual())                              # [3] Point-[F]
area_pull = (corners.adjugate() * turned_sides).sum(axis=0) / 2              # Point-[V]

# Pairing each corner's simplex gradient with half every corner's turned side forms the cotangent Laplacian.
local_laplacian = (gradients[:, None] & turned_sides[None, :]) / 2           # [3, 3] Scalar+[F]
laplacian = (corners[:, None].adjugate() * spdiag(local_laplacian) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[V <- V]

In [ ]:
def plateau_states(
    topology: Topology, initial_vertices: Point, step_size: float, n_steps: int,
) -> Iterator[tuple[Point, Scalar, Curvature]]:
    """Yields the surface (`Point-[V]`), total area (`Scalar+`), and interior mean curvature (`Curvature-[V]`) at each damped harmonic step with the boundary wire held."""
    corners = topology.incidence[2, 0]                                       # [3] Scalar+[F <- V]
    interior = topology.reading(topology.interior[0], 0)                     # Scalar+[V_interior <- V]
    exterior = topology.reading(topology.exterior[0], 0)                     # Scalar+[V_exterior <- V]
    wire = exterior * initial_vertices                                       # Point-[V_exterior]
    vertices = initial_vertices                                              # Point-[V]
    for _ in range(n_steps + 1):
        sides = topology.boundary[2, 1] * (topology.d[0] * vertices)         # [3] Point-[F]
        patches = (sides[0] & sides[1]) / 2                                  # Patch-[F]
        areas = patches.norm()                                               # Scalar+[F]
        normals = patches / areas                                            # Patch-[F]
        # Each corner's simplex gradient vanishes along the facing side and rises to one at the corner.
        gradients = (sides | patches.inverse()) / 2                          # [3] Plane+[F]
        # Turning each facing side by a quarter turn in the triangle's plane gives twice that corner's area gradient.
        turned_sides = sides.commutator(normals.dual())                      # [3] Point-[F]
        # Pairing each corner's simplex gradient with half every corner's turned side forms the cotangent Laplacian.
        local_laplacian = (gradients[:, None] & turned_sides[None, :]) / 2   # [3, 3] Scalar+[F]
        laplacian = (corners[:, None].adjugate() * spdiag(local_laplacian) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[V <- V]
        area_pull = interior * (corners.adjugate() * turned_sides).sum(axis=0) / 2  # Point-[V_interior]
        # Dividing a vertex's area pull by twice its third of the surrounding patches extracts the mean curvature.
        vertex_patches = interior * (corners.mean(axis=0).adjugate() * patches)  # Patch-[V_interior]
        mean_curvature = (area_pull & vertex_patches.inverse()) / 2          # Curvature-[V_interior]
        yield vertices, areas.sites.sum(), interior.adjugate() * mean_curvature
        # Solving for vanishing pull on the current triangles with the wire held gives the harmonic target surface.
        wire_pull = (interior * laplacian * exterior.adjugate()) * wire      # Point-[V_interior]
        harmonic_interior = ((interior * laplacian * interior.adjugate()) * Point).solve(-wire_pull)  # Point-[V_interior]
        harmonic = interior.adjugate() * harmonic_interior + exterior.adjugate() * wire  # Point-[V]
        # Taking a damped step towards the harmonic surface keeps the triangles and Laplacian well-conditioned.
        vertices = (1 - step_size) * vertices + step_size * harmonic         # Point-[V]


relaxation_states = list(plateau_states(
    plateau_topology, initial_vertices, relaxation_step_size, n_relaxation_steps,
))
show_animation(
    render.plateau_animation(plateau_topology, flat_vertices, relaxation_states),
    "minimal_surfaces_plateau",
    frame_duration_ms,
    n_frames=n_relaxation_steps + 1,
    forward_then_back=True,
)

The coral tint marks the magnitude of the interior mean curvature, which fades as the domed sheet sinks onto the saddle-shaped soap film spanning the four-arch wire. The teal rings and coloured radial spokes follow fixed curves in the planar disk domain.

In [ ]:
relaxation_areas = stack([area for _, area, _ in relaxation_states])         # [n_relaxation_steps + 1] Scalar+
_, _, initial_curvature = relaxation_states[0]                               # Point-[V], Scalar+, Curvature-[V]
_, _, final_curvature = relaxation_states[-1]                                # Point-[V], Scalar+, Curvature-[V]
max_curvatures = stack((initial_curvature, final_curvature)).norm().batch().max(axis=-1)  # [2] Scalar+
print("Surface area, initial and minimal:", relaxation_areas[[0, -1]].to_array())
print("Largest mean curvature, initial and minimal:", max_curvatures.to_array())

## Conjugates from boundary sums

Up to a global translation, the harmonic conjugate satisfies `laplacian * conjugate == boundary_flux`, where the right-hand side sums half the facing sides around each vertex and cancels at every interior vertex. Alongside the relaxed four-arch film, Enneper's minimal surface provides an analytical benchmark with a closed-form conjugate.

In [ ]:
enneper_radius = 0.8
n_enneper_divisions = 10

In [ ]:
def conjugate_minimal_surface(topology: Topology, vertices: Point) -> tuple[Point, Point]:
    """Recovers the harmonic conjugate (`Point-[V]`) up to global translation from half the sum of the facing sides around each vertex (`Point-[V]`)."""
    corners = topology.incidence[2, 0]                                       # [3] Scalar+[F <- V]
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)             # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Patch-[F]
    areas = patches.norm()                                                   # Scalar+[F]
    normals = patches / areas                                                # Patch-[F]
    gradients = (sides | patches.inverse()) / 2                              # [3] Plane+[F]
    turned_sides = sides.commutator(normals.dual())                          # [3] Point-[F]
    local_laplacian = (gradients[:, None] & turned_sides[None, :]) / 2       # [3, 3] Scalar+[F]
    laplacian = (corners[:, None].adjugate() * spdiag(local_laplacian) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[V <- V]
    # Integrating the quarter-turned differential against the simplex gradients leaves half the bare facing side on each triangle.
    boundary_flux = (corners.adjugate() * sides).sum(axis=0) / 2             # Point-[V]
    # Because the Laplacian annihilates global translations, a least-squares solve selects one conjugate representative.
    conjugate = (laplacian * Point).lstsq(boundary_flux)                     # Point-[V]
    return conjugate, boundary_flux


minimal_vertices, _, _ = relaxation_states[-1]
plateau_conjugate, plateau_flux = conjugate_minimal_surface(plateau_topology, minimal_vertices)  # Point-[V], Point-[V]
enneper_topology, enneper_flat, enneper_vertices, exact_enneper_conjugate = scenarios.enneper_disk(
    enneper_radius, n_enneper_divisions,
)                                                                            # Topology, Point-[V], Point-[V], Point-[V]
enneper_conjugate, enneper_flux = conjugate_minimal_surface(enneper_topology, enneper_vertices)  # Point-[V], Point-[V]

save_figure(
    render.conjugates_figure(
        plateau_topology, flat_vertices, minimal_vertices, plateau_conjugate,
        enneper_topology, enneper_flat, enneper_vertices, enneper_conjugate,
    ),
    "minimal_surfaces_conjugates",
);

The top row shows the relaxed soap film on the left and its conjugate on the right, where the four arches turn into four diagonally shifted lobes; the bottom row shows Enneper's surface and its conjugate, Enneper's surface turned by half a right angle in space. Matched domain rings and radial spokes cross at right angles between each surface and its conjugate.

## Bonnet's family from catenoid to helicoid

Bonnet's family, `vertices * np.cos(angle) + conjugate * np.sin(angle)`, bends a catenoid into its conjugate, a helicoid, through minimal surfaces with the same normals, lengths and angles. The helicoid rises as it turns and would not close around the catenoid's neck, so the catenoid is slit along a meridian.

In [ ]:
catenoid_height = 1.15
catenoid_azimuth_span = 0.85 * np.pi
n_catenoid_heights = 22
n_catenoid_azimuths = 44
n_bonnet_frames = 36

In [ ]:
def bonnet_family(
    topology: Topology, vertices: Point, conjugate: Point, angles: np.ndarray,
) -> Iterator[tuple[Point, Patch]]:
    """Yields each isometric minimal surface (`Point-[V]`) and its vertex Gauss normals (`Patch-[V]`) along Bonnet's family between a minimal surface and its harmonic conjugate."""
    corners = topology.incidence[2, 0]                                       # [3] Scalar+[F <- V]
    for angle in angles:
        surface = vertices * np.cos(angle) + conjugate * np.sin(angle)       # Point-[V]
        sides = topology.boundary[2, 1] * (topology.d[0] * surface)          # [3] Point-[F]
        normals = (corners.adjugate() * (sides[0] & sides[1])).sum(axis=0).normalized()  # Patch-[V]
        yield surface, normals


catenoid_topology, catenoid_vertices, exact_helicoid, catenoid_rings, catenoid_meridians = scenarios.slit_catenoid(
    catenoid_height, catenoid_azimuth_span, n_catenoid_heights, n_catenoid_azimuths,
)                                                                            # Topology, Point-[V], Point-[V], [n_catenoid_heights, n_catenoid_azimuths], [n_catenoid_azimuths, n_catenoid_heights]
helicoid_vertices, catenoid_flux = conjugate_minimal_surface(catenoid_topology, catenoid_vertices)  # Point-[V], Point-[V]
bonnet_angles = np.linspace(0.0, np.pi / 2, n_bonnet_frames)
bonnet_states = list(bonnet_family(catenoid_topology, catenoid_vertices, helicoid_vertices, bonnet_angles))
show_animation(
    render.bonnet_animation(catenoid_topology, catenoid_rings, catenoid_meridians, bonnet_states),
    "minimal_surfaces_bonnet",
    frame_duration_ms,
    n_frames=n_bonnet_frames,
    forward_then_back=True,
)

The teal rings and coral meridians of the slit catenoid slide into the helices and rulings of the helicoid without changing their lengths or the angles at which they cross. The gold pins mark the vertex Gauss normals, which translate with their vertices while keeping fixed directions in space.

In [ ]:
# checks
# Each turned side equals twice its corner's area-weighted simplex gradient, and their vertex sum matches the Laplacian's pull.
assert ((turned_sides - 2 * areas * gradients.dual()).norm() < 1e-10).all()
area_gradient = (corners.adjugate() * (areas * gradients)).sum(axis=0)       # Plane+[V]
assert ((area_pull - area_gradient.dual()).norm() < 1e-10).all()
assert ((area_pull - laplacian * initial_vertices).norm() < 1e-10).all()

# Surface area decreases monotonically across relaxation and interior mean curvature decays by an order of magnitude.
assert (relaxation_areas[1:] < relaxation_areas[:-1]).all()
assert (final_curvature.norm() < 0.1 * initial_curvature.norm().batch().max()).all()

# Facing sides cancel across shared interior edges, leaving boundary sums alone.
assert ((plateau_topology.reading(plateau_topology.interior[0], 0) * plateau_flux).norm() < 1e-10).all()
assert ((enneper_topology.reading(enneper_topology.interior[0], 0) * enneper_flux).norm() < 1e-10).all()
assert ((catenoid_topology.reading(catenoid_topology.interior[0], 0) * catenoid_flux).norm() < 1e-10).all()

# Recovered conjugates match the closed-form conjugates up to global translation.
centred_enneper = enneper_conjugate - enneper_conjugate.sites.mean()         # Point-[V]
centred_exact_enneper = exact_enneper_conjugate - exact_enneper_conjugate.sites.mean()  # Point-[V]
assert ((centred_enneper - centred_exact_enneper).norm() < 2e-2).all()
centred_helicoid = helicoid_vertices - helicoid_vertices.sites.mean()        # Point-[V]
centred_exact_helicoid = exact_helicoid - exact_helicoid.sites.mean()        # Point-[V]
assert ((centred_helicoid - centred_exact_helicoid).norm() < 2e-2).all()

# Bonnet's family preserves edge lengths and vertex Gauss normals between catenoid and helicoid.
catenoid_frame, catenoid_normals = bonnet_states[0]
helicoid_frame, helicoid_normals = bonnet_states[-1]
catenoid_lengths = (catenoid_topology.d[0] * catenoid_frame).norm()          # Scalar+[E]
helicoid_lengths = (catenoid_topology.d[0] * helicoid_frame).norm()          # Scalar+[E]
assert (((helicoid_lengths - catenoid_lengths).abs() / catenoid_lengths) < 1e-2).all()
assert ((helicoid_normals - catenoid_normals).norm() < 5e-2).all()